# 初期セットアップ（ステップ２）
このノートブックでは、サンプルコードの実行環境を初めて使用する際の設定のステップ２を行います。  
ステップ２では、ブロックチェーンへの接続の設定を行います。  

## 設定項目（実行前に、必要に応じて書き換えてください）

chainIDとpeerURLを設定します。  
試使用環境とは別のブロックチェーンに接続する場合は、下のセルを書き換えてください。

In [1]:
var chainID = "trial.dncware-blockchain.biz";
var peerURL = "https://trial1.dncware-blockchain.biz";

**（オプショナル）**  
プロキシの設定をします。
ブロックチェーンへの接続にプロキシを経由する環境の場合の設定です。  
ネットワークがプロキシ環境ではない場合は、設定の必要はありません。(コメントアウトしたままとします）  
プロキシを設定する場合には、先頭の//を削除し、プロキシのURLを適宜に設定します。

In [2]:
// var proxy = "http://proxy.mycompany.co.jp:8080";

**（オプショナル）**  
接続先のブロックチェーンの証明書の検証に用いるCertificate Authority(CA)の設定をします。  
通常は、この設定の必要はありません。(コメントアウトしたままとします）  
この設定は、プライベート環境にブロックチェーンを配置する場合や、セキュリティ機能が強化されたプロキシを経由する場合などでの利用を想定しています。  
CAを設定する場合は、先頭の//を削除し、CAの証明書ファイルへのパスを設定します。

In [3]:
// var CA = "full path to certificate";

## 必要なライブラリの読み込みと設定の反映

In [4]:
var { api, config_proxy_ca } = await import('../lib/load-blockchain-api.mjs');
var { adminWallet } = await import('../lib/notebook-util.mjs');

proxyとCAの設定をライブラリに反映します。

In [5]:
var proxy, CA;
await config_proxy_ca(proxy, CA);

## 接続確認

ブロックチェーンへの基本的な接続を確認します。  
OKと表示されれば合格です。

In [6]:
var rpc = new api.RPC(chainID);
rpc.connect(peerURL);
try {
    await rpc.fetchBlock();
    console.log('OK');
    var success_1 = true;
} catch (err){
    console.error('NG');
    console.error(err);
    var success_1 = false;
}

OK


ブロックチェーンへのウォレットを使った接続を確認します。  
OKと表示されれば合格です。

In [7]:
var rpc = new api.RPC(chainID);
rpc.connect(peerURL);
try {
    await rpc.call(adminWallet, 'c1query', { type: 'a_wallet' });
    console.log('OK');
    var success_2 = true;
} catch (err){
    console.error('NG');
    console.error(err);
    var success_2 = false;
}

OK


## ピア構成情報の取得

ピア構成情報を表す文字列cnfstrを取得します。  
ピア構成情報は、ピアの公開鍵の情報を含み、クライアント側での検証の基礎となります。  

In [8]:
var { V } = await rpc.fetchCnfstr();
var { cnfstr } = await rpc.fetchCnfstr(V);

## 設定ファイルの保存
上記の接続確認が成功した場合に、設定を設定ファイル(etc/config.json)に書き込みます。  

In [9]:
if(success_1 !== true || success_2 !== true) throw '接続確認が成功していません';
var fs = await import('node:fs');
var path = await import('node:path');
var { default: package_root } = await import('../lib/get-package-root.mjs');
var config_filename = path.join(package_root, 'etc', 'config.json');
if(!fs.existsSync(config_filename)) throw '設定ファイルが存在しません:'+config_filename
var config = JSON.parse(fs.readFileSync(config_filename, 'utf8'));
Object.assign(config, { chainID, peerURL, proxy, CA, cnfstr });
fs.writeFileSync(config_filename, JSON.stringify(config, null, 1), 'utf8');
console.log('DONE');

DONE
